# LightRAG Hybrid Search Demo
## Vector + BM25 Keyword Search with RRF Fusion

This notebook demonstrates the hybrid search feature added to LightRAG.
We test the BM25 index, RRF fusion, and compare results.

## 1. BM25Index Unit Test

In [1]:
import sys
sys.path.insert(0, '..')

from lightrag.bm25_index import BM25Index, reciprocal_rank_fusion

In [2]:
# Build a BM25 index with sample documents
documents = {
    "doc_gpt4o": "GPT-4o is a multimodal AI model developed by OpenAI that can process text, images, and audio.",
    "doc_llama": "LLaMA is an open-source large language model released by Meta AI for research purposes.",
    "doc_bert": "BERT is a transformer-based model developed by Google for natural language understanding tasks.",
    "doc_rag": "Retrieval Augmented Generation (RAG) combines document retrieval with language model generation.",
    "doc_lightrag": "LightRAG is a graph-based RAG system developed by HKUDS that uses knowledge graphs for retrieval.",
    "doc_bm25": "BM25 is a ranking function used in information retrieval based on term frequency and inverse document frequency.",
    "doc_vector": "Vector similarity search uses embeddings to find semantically similar documents in high-dimensional space.",
    "doc_hybrid": "Hybrid search combines keyword-based BM25 search with vector similarity for improved retrieval accuracy.",
    "doc_rrf": "Reciprocal Rank Fusion (RRF) merges results from multiple retrieval systems using rank-based scoring.",
    "doc_kg": "Knowledge graphs represent entities and their relationships as nodes and edges in a graph structure.",
}

bm25 = BM25Index()
bm25.build(documents)
print(f"Index built: {bm25.is_built}")
print(f"Corpus size: {len(bm25.corpus_ids)}")

INFO: BM25 index built with 10 documents


Index built: True
Corpus size: 10


In [3]:
# Test 1: Exact keyword match - "GPT-4o"
print("=" * 60)
print("Query: 'GPT-4o'")
print("=" * 60)
results = bm25.query("GPT-4o", top_k=5)
for r in results:
    print(f"  {r['id']:20s}  score={r['score']:.4f}")
print()

# Test 2: Technical term - "BM25"
print("=" * 60)
print("Query: 'BM25 ranking'")
print("=" * 60)
results = bm25.query("BM25 ranking", top_k=5)
for r in results:
    print(f"  {r['id']:20s}  score={r['score']:.4f}")
print()

# Test 3: Proper noun - "HKUDS"
print("=" * 60)
print("Query: 'HKUDS'")
print("=" * 60)
results = bm25.query("HKUDS", top_k=5)
for r in results:
    print(f"  {r['id']:20s}  score={r['score']:.4f}")
print()

# Test 4: Semantic query - "document retrieval methods"
print("=" * 60)
print("Query: 'document retrieval methods'")
print("=" * 60)
results = bm25.query("document retrieval methods", top_k=5)
for r in results:
    print(f"  {r['id']:20s}  score={r['score']:.4f}")

Query: 'GPT-4o'
  doc_gpt4o             score=3.4488

Query: 'BM25 ranking'
  doc_bm25              score=2.8677
  doc_hybrid            score=1.2506

Query: 'HKUDS'
  doc_lightrag          score=1.7752

Query: 'document retrieval methods'
  doc_rag               score=1.3801
  doc_bm25              score=1.1433


## 2. RRF (Reciprocal Rank Fusion) Demo

In [4]:
# Simulate vector search results (sorted by cosine similarity)
vector_results = [
    {"id": "doc_rag", "score": 0.92, "entity_name": "RAG"},
    {"id": "doc_hybrid", "score": 0.88, "entity_name": "Hybrid Search"},
    {"id": "doc_vector", "score": 0.85, "entity_name": "Vector Search"},
    {"id": "doc_kg", "score": 0.82, "entity_name": "Knowledge Graph"},
    {"id": "doc_bert", "score": 0.78, "entity_name": "BERT"},
]

# Simulate BM25 search results (sorted by BM25 score)
bm25_results = [
    {"id": "doc_lightrag", "score": 8.5},
    {"id": "doc_rag", "score": 6.2},
    {"id": "doc_bm25", "score": 4.1},
    {"id": "doc_hybrid", "score": 3.8},
]

print("Vector Results (by cosine similarity):")
for i, r in enumerate(vector_results, 1):
    print(f"  #{i}: {r['id']:20s} (score={r['score']})")

print("\nBM25 Results (by keyword match):")
for i, r in enumerate(bm25_results, 1):
    print(f"  #{i}: {r['id']:20s} (score={r['score']})")

# Apply RRF fusion
fused = reciprocal_rank_fusion(vector_results, bm25_results, k=60)

print("\nRRF Fused Results:")
for i, r in enumerate(fused, 1):
    in_vec = any(v['id'] == r['id'] for v in vector_results)
    in_bm25 = any(b['id'] == r['id'] for b in bm25_results)
    source = "BOTH" if (in_vec and in_bm25) else ("vector" if in_vec else "bm25")
    print(f"  #{i}: {r['id']:20s} [{source}]")

Vector Results (by cosine similarity):
  #1: doc_rag              (score=0.92)
  #2: doc_hybrid           (score=0.88)
  #3: doc_vector           (score=0.85)
  #4: doc_kg               (score=0.82)
  #5: doc_bert             (score=0.78)

BM25 Results (by keyword match):
  #1: doc_lightrag         (score=8.5)
  #2: doc_rag              (score=6.2)
  #3: doc_bm25             (score=4.1)
  #4: doc_hybrid           (score=3.8)

RRF Fused Results:
  #1: doc_rag              [BOTH]
  #2: doc_hybrid           [BOTH]
  #3: doc_lightrag         [bm25]
  #4: doc_vector           [vector]
  #5: doc_bm25             [bm25]
  #6: doc_kg               [vector]
  #7: doc_bert             [vector]


## 3. RRF Score Calculation Detail

In [5]:
import pandas as pd

k = 60
all_docs = set()
vec_rank = {}
bm25_rank = {}

for i, r in enumerate(vector_results, 1):
    vec_rank[r['id']] = i
    all_docs.add(r['id'])

for i, r in enumerate(bm25_results, 1):
    bm25_rank[r['id']] = i
    all_docs.add(r['id'])

rows = []
for doc in all_docs:
    vr = vec_rank.get(doc)
    br = bm25_rank.get(doc)
    vec_score = 1/(k + vr) if vr else 0
    bm25_score = 1/(k + br) if br else 0
    total = vec_score + bm25_score
    rows.append({
        'Document': doc,
        'Vec Rank': vr or '-',
        'BM25 Rank': br or '-',
        'Vec RRF': f'{vec_score:.6f}' if vr else '-',
        'BM25 RRF': f'{bm25_score:.6f}' if br else '-',
        'Total RRF': f'{total:.6f}',
        'Source': 'BOTH' if (vr and br) else ('vector' if vr else 'bm25'),
    })

df = pd.DataFrame(rows).sort_values('Total RRF', ascending=False).reset_index(drop=True)
df.index = df.index + 1
df.index.name = 'Final Rank'
df

,Document,Vec Rank,BM25 Rank,Vec RRF,BM25 RRF,Total RRF,Source
Final Rank,,,,,,,
1,doc_rag,1,2,0.016393,0.016129,0.032522,BOTH
2,doc_hybrid,2,4,0.016129,0.015625,0.031754,BOTH
3,doc_lightrag,-,1,-,0.016393,0.016393,bm25
4,doc_vector,3,-,0.015873,-,0.015873,vector
5,doc_bm25,-,3,-,0.015873,0.015873,bm25
6,doc_kg,4,-,0.015625,-,0.015625,vector
7,doc_bert,5,-,0.015385,-,0.015385,vector


## 4. Visualization: Vector vs BM25 vs Hybrid Rankings

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Compare rankings across methods
docs_order = [r['id'] for r in fused]
vec_ranks = [vec_rank.get(d, len(vector_results)+1) for d in docs_order]
bm25_ranks = [bm25_rank.get(d, len(bm25_results)+1) for d in docs_order]
fused_ranks = list(range(1, len(docs_order)+1))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(docs_order))
width = 0.25

bars1 = ax.bar(x - width, vec_ranks, width, label='Vector Rank', color='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
bars2 = ax.bar(x, bm25_ranks, width, label='BM25 Rank', color='#FFCCBC', edgecolor='#E64A19', linewidth=1.5)
bars3 = ax.bar(x + width, fused_ranks, width, label='RRF Fused Rank', color='#C8E6C9', edgecolor='#2E7D32', linewidth=1.5)

ax.set_xlabel('Documents', fontsize=12)
ax.set_ylabel('Rank (lower is better)', fontsize=12)
ax.set_title('Ranking Comparison: Vector vs BM25 vs Hybrid (RRF)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([d.replace('doc_', '') for d in docs_order], rotation=30, ha='right')
ax.legend(fontsize=10)
ax.invert_yaxis()
ax.set_ylim(max(max(vec_ranks), max(bm25_ranks)) + 0.5, 0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Ranking comparison chart saved to images/ranking_comparison.png")

Ranking comparison chart saved to images/ranking_comparison.png


## 5. BM25 Advantage: Proper Noun / Keyword Queries

In [7]:
# Demonstrate where BM25 excels vs vector search
test_queries = [
    ("GPT-4o", "Exact proper noun - BM25 strength"),
    ("HKUDS", "Unique identifier - BM25 strength"),
    ("BM25 ranking function", "Technical term - BM25 strength"),
    ("how do AI systems understand language", "Semantic query - Vector strength"),
    ("graph-based information retrieval", "Mixed query - Hybrid advantage"),
]

print(f"{'Query':<45} {'Top BM25 Result':<25} {'BM25 Score':>10}")
print("=" * 82)
for query, description in test_queries:
    results = bm25.query(query, top_k=3)
    if results:
        top = results[0]
        print(f"{query:<45} {top['id']:<25} {top['score']:>10.4f}")
    else:
        print(f"{query:<45} {'(no results)':<25} {'N/A':>10}")
    print(f"  -> {description}")
    print()

Query                                         Top BM25 Result           BM25 Score
GPT-4o                                        doc_gpt4o                     3.4488
  -> Exact proper noun - BM25 strength

HKUDS                                         doc_lightrag                  1.7752
  -> Unique identifier - BM25 strength

BM25 ranking function                         doc_bm25                      4.5921
  -> Technical term - BM25 strength

how do AI systems understand language         doc_llama                     1.9678
  -> Semantic query - Vector strength

graph-based information retrieval             doc_bm25                      1.7244
  -> Mixed query - Hybrid advantage



## 6. Edge Cases

In [8]:
# Test edge cases
print("Edge Case 1: Empty query")
results = bm25.query("", top_k=5)
print(f"  Results: {results}")
print()

print("Edge Case 2: Query with no matches")
results = bm25.query("xyznonexistent123", top_k=5)
print(f"  Results: {results}")
print()

print("Edge Case 3: Empty index")
empty_idx = BM25Index()
print(f"  is_built: {empty_idx.is_built}")
results = empty_idx.query("test", top_k=5)
print(f"  Results: {results}")
print()

print("Edge Case 4: RRF with empty lists")
fused = reciprocal_rank_fusion([], [])
print(f"  Results: {fused}")
print()

print("Edge Case 5: RRF with one empty list")
vec_only = [{"id": "a", "score": 0.9}, {"id": "b", "score": 0.8}]
fused = reciprocal_rank_fusion(vec_only, [])
print(f"  Results: {[r['id'] for r in fused]}")
print()

print("All edge cases passed!")

Edge Case 1: Empty query
  Results: []

Edge Case 2: Query with no matches
  Results: []

Edge Case 3: Empty index
  is_built: False
  Results: []

Edge Case 4: RRF with empty lists
  Results: []

Edge Case 5: RRF with one empty list
  Results: ['a', 'b']

All edge cases passed!


## 7. Configuration Check

In [9]:
from lightrag.addon_params import default_addon_params

defaults = default_addon_params()
print("Default addon_params:")
for k, v in defaults.items():
    if k != 'chunker':  # Skip verbose chunker config
        print(f"  {k}: {v}")

print(f"\nenable_hybrid_search default: {defaults.get('enable_hybrid_search')}")
assert defaults.get('enable_hybrid_search') == False, "Default should be False"
print("Configuration check passed!")

Default addon_params:
  language: English
  entity_type_prompt_file: 
  enable_hybrid_search: False

enable_hybrid_search default: False
Configuration check passed!


## Summary

### Key Findings

1. **BM25 excels at exact keyword matching** - proper nouns, technical terms, unique identifiers
2. **RRF fusion correctly boosts documents found by both methods** - rank-based, no score normalization needed
3. **Edge cases handled gracefully** - empty queries, no matches, empty indices
4. **Backward compatible** - `enable_hybrid_search` defaults to `False`

### Expected Improvement

| Query Type | Vector Only | Hybrid |
|---|---|---|
| Proper nouns | Approximate match | **Exact match** |
| Short queries | Low quality embedding | **BM25 IDF effective** |
| Technical terms | Semantic confusion | **Token-level match** |
| Long queries | Effective | Vector primary + BM25 supplement |